# Resource Desert Analysis & Optimization
## UNF CodeforAwhile Datathon 2025 — Problem Case 1

This notebook orchestrates the full five-stage pipeline:

| Stage | Module | Purpose |
|---|---|---|
| 1 | `ingestion` | Load all 9 raw CSV datasets |
| 2 | `cleaning` | Standardise ZIPs, fix dtypes, clip rates |
| 3 | `features` | Merge, compute Desert Score |
| 4 | `models` | Gap-closure simulation & intervention ranking |
| 5 | `visualization` | Charts, choropleth map |

**Key questions answered:**
1. Which Jacksonville ZIP codes are the most underserved resource deserts?
2. How do resource deserts correlate with health outcomes?
3. Which single intervention would deliver the greatest improvement?

In [1]:
import warnings
import sys
from pathlib import Path

import matplotlib
matplotlib.use('Agg')  # headless backend for reproducible output
import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings('ignore')

# Ensure src/ is importable from the notebook
repo_root = Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src import config as cfg
from src.ingestion import load_raw_datasets
from src.cleaning import clean_datasets
from src.features import (
    merge_datasets,
    compute_desert_score,
    compute_health_outcome_correlation,
    filter_by_service_category,
)
from src.models import run_gap_closure_simulation, rank_interventions
from src.visualization import (
    plot_desert_scores_bar_chart,
    create_choropleth_map,
    plot_preventative_vs_outcome,
    plot_intervention_impact_heatmap,
    plot_category_view,
)

print('Environment ready.')

Environment ready.


---
## Stage 1 — Data Ingestion

Load all 9 raw CSV/XLSX source files from `data/raw/`.  No transformations are applied here.

In [2]:
raw = load_raw_datasets()

print(f"Loaded {len(raw)} datasets:")
for key, df in raw.items():
    print(f"  {key:30s}  {df.shape[0]:>5} rows × {df.shape[1]:>3} cols")

2026-03-28 22:51:16,324 INFO [src.ingestion] Loaded 'cdc_places': 37 rows × 18 cols from CDCPlaces.csv


2026-03-28 22:51:16,325 INFO [src.ingestion] Loaded 'census_demographics': 37 rows × 28 cols from Census-Demographics.csv


2026-03-28 22:51:16,326 INFO [src.ingestion] Loaded 'census_housing_poverty': 37 rows × 12 cols from Census-Housing&Poverty.csv


2026-03-28 22:51:16,326 INFO [src.ingestion] Loaded 'fema': 37 rows × 9 cols from FEMA.csv


2026-03-28 22:51:16,327 INFO [src.ingestion] Loaded 'healthcare_access': 37 rows × 13 cols from HealthCareAccess.csv


...

2026-03-28 22:51:16,329 INFO [src.ingestion] Loaded 'social_vulnerability': 37 rows × 6 cols from SocialVulnerabilityIndex.csv


2026-03-28 22:51:16,330 INFO [src.ingestion] Loaded 'usda_food_access': 37 rows × 16 cols from USDA-FoodAccess.csv


2026-03-28 22:51:16,384 INFO [src.ingestion] Loaded 'metadata': 81 rows × 4 cols from Metadata.xlsx


2026-03-28 22:51:16,384 INFO [src.ingestion] Loaded 10 datasets successfully.


Loaded 10 datasets:
  cdc_places                         37 rows ×  18 cols
  census_demographics                37 rows ×  28 cols
  census_housing_poverty             37 rows ×  12 cols
  fema                               37 rows ×   9 cols
  healthcare_access                  37 rows ×  13 cols
  healthcare_workers                 37 rows ×   8 cols
  parks                              37 rows ×   7 cols
  social_vulnerability               37 rows ×   6 cols
  usda_food_access                   37 rows ×  16 cols
  metadata                           81 rows ×   4 cols


---
## Stage 2 — Data Cleaning

- Standardise the ZIP column to `zip_code` (str, 5-char, zero-padded)
- Fix column dtypes
- Clip rate/proportion columns to `[0.0, 1.0]`
- Drop duplicates; log before/after row counts

In [3]:
cleaned = clean_datasets(raw)

print("Post-cleaning shapes:")
for key, df in cleaned.items():
    print(f"  {key:30s}  {df.shape[0]:>5} rows × {df.shape[1]:>3} cols")

2026-03-28 22:51:16,387 INFO [src.cleaning] Cleaning 'cdc_places': 37 rows before.


2026-03-28 22:51:16,391 INFO [src.cleaning] Cleaning 'cdc_places': 37 rows after.


2026-03-28 22:51:16,391 INFO [src.cleaning] Cleaning 'census_demographics': 37 rows before.


2026-03-28 22:51:16,395 INFO [src.cleaning] Cleaning 'census_demographics': 37 rows after.


2026-03-28 22:51:16,395 INFO [src.cleaning] Cleaning 'census_housing_poverty': 37 rows before.


...

2026-03-28 22:51:16,406 INFO [src.cleaning] Cleaning 'social_vulnerability': 37 rows after.


2026-03-28 22:51:16,406 INFO [src.cleaning] Cleaning 'usda_food_access': 37 rows before.


2026-03-28 22:51:16,409 INFO [src.cleaning] Cleaning 'usda_food_access': 37 rows after.


2026-03-28 22:51:16,409 INFO [src.cleaning] Cleaning 'metadata': 81 rows before.


Post-cleaning shapes:
  cdc_places                         37 rows ×  19 cols
  census_demographics                37 rows ×  29 cols
  census_housing_poverty             37 rows ×  13 cols
  fema                               37 rows ×  10 cols
  healthcare_access                  37 rows ×  14 cols
  healthcare_workers                 37 rows ×   9 cols
  parks                              37 rows ×   8 cols
  social_vulnerability               37 rows ×   7 cols
  usda_food_access                   37 rows ×  17 cols
  metadata                           81 rows ×   4 cols


---
## Stage 3 — Feature Engineering

### 3a. Merge datasets

Outer-join all 9 cleaned datasets on `zip_code`, filter to Duval County ZIPs, impute null supply metrics with column medians.

In [4]:
merged = merge_datasets(cleaned)
print(f"Merged dataset: {merged.shape[0]} ZIP codes × {merged.shape[1]} columns")
merged.head(3)

2026-03-28 22:51:16,418 INFO [src.features] Merged all datasets: 37 rows before population filter.


2026-03-28 22:51:16,419 INFO [src.features] Dropped 2 ZIPs with population == 0 or missing population.


2026-03-28 22:51:16,421 INFO [src.features] Imputed 1 nulls in 'Primary Care Physician Ratio (2025)' with median 979.8850.


2026-03-28 22:51:16,423 INFO [src.features] Saved merged dataset: 35 rows → /Users/amitrajpurkar/workspace/pyprojects/unf_datathons/prob1-resource-desert/data/processed/merged_jacksonville.csv


Merged dataset: 35 ZIP codes × 28 columns


,zip_code,Total Population (2020-2024),People Below Poverty Level (2020-2024),Median Household Income (2020-2024),Poor Mental Health Among Adults (2023),Obesity Among Adults (2023),No Leisure-Time Physical Activity Among Adults (2023),Life Expectancy at Birth (2010-2015),Primary Care Physician Ratio (2025),Primary Care Nurse Practitioner Ratio (2025),...,People 1 Mile Urban/20 Miles Rural with Low Access to Healthy Food (2019),Low Income People (USDA) (2019),Social Vulnerability Index Within the State (2022),Social Vulnerability Index Highly Vulnerable Factors Within the State (2022),Environmental Hazard Community Resilience Score (2025),Environmental Hazard Expected Annual Loss Total (2025),Social Vulnerability to Environmental Hazards (2024),poverty_rate,uninsured_rate,food_low_access_pct
0,32202,6023,1400,34825.0,25.3,47.5,45.4,73.0944,354.29,463.31,...,54.0,2127,0.9311,7.0,49.78,3447393.751,0.5672,0.232442,0.112568,0.008966
1,32204,9151,1733,65063.0,19.1,35.0,29.4,75.5328,84.73,65.36,...,1222.0,3147,0.6444,2.0,49.78,4758411.616,0.5139,0.189378,0.110261,0.133537
2,32205,29148,4540,64789.0,18.9,33.7,25.4,74.2022,1324.91,1267.30,...,4888.0,11875,0.5961,0.0,49.78,8929993.683,0.3958,0.155757,0.095067,0.167696


### 3b. Compute Desert Score

The **Resource Desert Score** (0–100, higher = more deprived) is computed as:

```
demand_factor  = normalise(poverty_rate × disease_burden_composite)
supply_gap_*   = normalised deprivation per resource type
desert_score   = normalise(Σ supply_gap_i × demand_factor) × 100
```

Ties are broken by `poverty_rate` descending.

In [5]:
desert_scores = compute_desert_score(merged)
print(f"Desert scores computed for {len(desert_scores)} ZIP codes.")
print(f"Score range: [{desert_scores[cfg.COL_DESERT_SCORE].min():.1f}, "
      f"{desert_scores[cfg.COL_DESERT_SCORE].max():.1f}]")

2026-03-28 22:51:16,436 INFO [src.features] Desert scores saved: 35 ZIPs → /Users/amitrajpurkar/workspace/pyprojects/unf_datathons/prob1-resource-desert/reports/outputs/desert_scores.csv


Desert scores computed for 35 ZIP codes.
Score range: [0.0, 100.0]


### Top-10 Most Underserved ZIP Codes

In [6]:
top10 = (
    desert_scores
    .nsmallest(10, cfg.COL_DESERT_RANK)
    [[cfg.COL_ZIP, cfg.COL_DESERT_RANK, cfg.COL_DESERT_SCORE,
      cfg.COL_TOP_GAP_CATEGORY, cfg.COL_TOTAL_POPULATION,
      cfg.COL_POVERTY_RATE]]
    .rename(columns={
        cfg.COL_ZIP: 'ZIP',
        cfg.COL_DESERT_RANK: 'Rank',
        cfg.COL_DESERT_SCORE: 'Desert Score',
        cfg.COL_TOP_GAP_CATEGORY: 'Top Gap',
        cfg.COL_TOTAL_POPULATION: 'Population',
        cfg.COL_POVERTY_RATE: 'Poverty Rate',
    })
    .set_index('Rank')
)
top10.style.format({'Desert Score': '{:.1f}', 'Poverty Rate': '{:.1%}'})

,ZIP,Desert Score,Top Gap,Population,Poverty Rate
Rank,,,,,
1,32209,100.0,healthcare,34657,37.6%
2,32254,96.6,insurance,13927,33.7%
3,32206,84.2,parks,17105,32.4%
4,32208,70.8,parks,32699,28.1%
5,32202,57.4,healthcare,6023,23.2%
6,32211,48.7,parks,36762,22.6%
7,32204,37.0,healthcare,9151,18.9%
8,32277,37.0,insurance,36338,17.8%
9,32210,35.8,parks,65729,16.5%


### Bar Chart — Top-10 Desert Scores

In [7]:
plot_desert_scores_bar_chart(desert_scores)
print(f"Saved → {cfg.FIGURE_BAR_CHART}")

2026-03-28 22:51:16,548 INFO [src.visualization] Saved bar chart → /Users/amitrajpurkar/workspace/pyprojects/unf_datathons/prob1-resource-desert/reports/figures/desert_scores_bar_chart.png


Saved → /Users/amitrajpurkar/workspace/pyprojects/unf_datathons/prob1-resource-desert/reports/figures/desert_scores_bar_chart.png


### Choropleth Map

The interactive map is saved as a standalone HTML file.  Open it in a browser to explore ZIP-level scores.

> **Note**: Requires `data/raw/jacksonville_zctas.geojson`.  The cell is skipped gracefully if the file is absent.

In [8]:
if cfg.GEOJSON_PATH.exists():
    create_choropleth_map(desert_scores)
    print(f"Map saved → {cfg.CHOROPLETH_MAP_PATH}")
    from IPython.display import IFrame
    display(IFrame(str(cfg.CHOROPLETH_MAP_PATH), width=900, height=500))
else:
    print(f"GeoJSON not found at {cfg.GEOJSON_PATH} — skipping choropleth.")

2026-03-28 22:51:17,046 INFO [src.visualization] Saved choropleth map → /Users/amitrajpurkar/workspace/pyprojects/unf_datathons/prob1-resource-desert/reports/outputs/resource_desert_map.html


Map saved → /Users/amitrajpurkar/workspace/pyprojects/unf_datathons/prob1-resource-desert/reports/outputs/resource_desert_map.html


---
## Stage 4 — Health Outcome Correlation (US2)

Quantify the Pearson correlation between each preventative asset metric and health outcome metric.

In [9]:
corr_df = compute_health_outcome_correlation(merged)
print("Pearson correlation — Preventative Assets vs Health Outcomes:")

# Pivot to wide matrix (assets as rows, outcomes as columns)
corr_matrix = corr_df.pivot(index="asset", columns="outcome", values="pearson_r")
corr_matrix.index.name = "Asset"
corr_matrix.columns.name = "Outcome"
corr_matrix.style.background_gradient(cmap="RdYlGn", vmin=-1, vmax=1).format("{:.3f}")

2026-03-28 22:51:17,050 INFO [src.features] Correlation primary_care_ratio × poor_mental_health_pct: r = -0.0260


2026-03-28 22:51:17,051 INFO [src.features] Correlation primary_care_ratio × obesity_pct: r = 0.1135


2026-03-28 22:51:17,051 INFO [src.features] Correlation food_low_access_pct × poor_mental_health_pct: r = 0.2930


2026-03-28 22:51:17,052 INFO [src.features] Correlation food_low_access_pct × obesity_pct: r = -0.2189


2026-03-28 22:51:17,052 INFO [src.features] Correlation park_coverage_pct × poor_mental_health_pct: r = -0.1979


2026-03-28 22:51:17,053 INFO [src.features] Correlation park_coverage_pct × obesity_pct: r = -0.1150


2026-03-28 22:51:17,053 INFO [src.features] Correlation uninsured_rate × poor_mental_health_pct: r = -0.0613


2026-03-28 22:51:17,054 INFO [src.features] Correlation uninsured_rate × obesity_pct: r = 0.6064


Pearson correlation — Preventative Assets vs Health Outcomes:


Outcome,obesity_pct,poor_mental_health_pct
Asset,,
food_low_access_pct,-0.219,0.293
park_coverage_pct,-0.115,-0.198
primary_care_ratio,0.114,-0.026
uninsured_rate,0.606,-0.061


In [10]:
# Plot the strongest association
plot_preventative_vs_outcome(
    merged,
    asset_col=cfg.COL_PRIMARY_CARE_RATIO,
    outcome_col=cfg.COL_POOR_MENTAL_HEALTH,
)
print(f"Scatter plot saved → {cfg.FIGURE_HEALTH_SCATTER}")

2026-03-28 22:51:17,153 INFO [src.visualization] Saved health scatter → /Users/amitrajpurkar/workspace/pyprojects/unf_datathons/prob1-resource-desert/reports/figures/preventative_asset_vs_health_outcome.png


Scatter plot saved → /Users/amitrajpurkar/workspace/pyprojects/unf_datathons/prob1-resource-desert/reports/figures/preventative_asset_vs_health_outcome.png


---
## Stage 5 — Gap-Closure Simulation (US3)

For each of the **top-5 most underserved ZIPs** × **4 resource types**, simulate setting the supply metric to the 25th-percentile target and compute the Desert Score improvement.

In [11]:
interventions = run_gap_closure_simulation(desert_scores, merged, top_n=5)
ranked = rank_interventions(interventions)

best = ranked[ranked['is_highest_impact']].iloc[0]
print("=" * 60)
print("HIGHEST-IMPACT INTERVENTION")
print("=" * 60)
print(f"  ZIP Code        : {best['zip_code']}")
print(f"  Resource Type   : {best['resource_type']}")
print(f"  Current Score   : {best['current_desert_score']:.1f}")
print(f"  Simulated Score : {best['simulated_desert_score']:.1f}")
print(f"  Improvement     : {best['score_improvement']:.1f} pts ({best['pct_improvement']:.1f}%)")
print(f"  Population      : {best['population_impacted']:,}")
print(f"\nFull recommendations → {cfg.INTERVENTION_RECOMMENDATIONS_PATH}")

2026-03-28 22:51:17,163 INFO [src.models] Gap-closure simulation complete: 20 interventions across top-5 ZIPs.


2026-03-28 22:51:17,167 INFO [src.models] Saved 20 intervention recommendations → /Users/amitrajpurkar/workspace/pyprojects/unf_datathons/prob1-resource-desert/reports/outputs/intervention_recommendations.json


2026-03-28 22:51:17,167 INFO [src.models] Highest-impact action: insurance_outreach in ZIP 32254 — 14.3% Desert Score improvement (96.63 → 82.79)


HIGHEST-IMPACT INTERVENTION
  ZIP Code        : 32254
  Resource Type   : insurance_outreach
  Current Score   : 96.6
  Simulated Score : 82.8
  Improvement     : 13.8 pts (14.3%)
  Population      : 13,927

Full recommendations → /Users/amitrajpurkar/workspace/pyprojects/unf_datathons/prob1-resource-desert/reports/outputs/intervention_recommendations.json


In [12]:
plot_intervention_impact_heatmap(ranked)
print(f"Heatmap saved → {cfg.FIGURE_INTERVENTION_HEATMAP}")

2026-03-28 22:51:17,262 INFO [src.visualization] Saved intervention heatmap → /Users/amitrajpurkar/workspace/pyprojects/unf_datathons/prob1-resource-desert/reports/figures/intervention_impact_heatmap.png


Heatmap saved → /Users/amitrajpurkar/workspace/pyprojects/unf_datathons/prob1-resource-desert/reports/figures/intervention_impact_heatmap.png


### Intervention Recommendations Summary

In [13]:
ranked[['zip_code', 'resource_type', 'score_improvement',
        'pct_improvement', 'population_impacted', 'is_highest_impact']].style.apply(
    lambda row: ['background-color: #ffe066' if row['is_highest_impact'] else '' for _ in row],
    axis=1
).format({'score_improvement': '{:.2f}', 'pct_improvement': '{:.1f}%'})

,zip_code,resource_type,score_improvement,pct_improvement,population_impacted,is_highest_impact
0,32254,insurance_outreach,13.84,14.3%,13927,True
1,32254,food_access,12.63,13.1%,13927,False
2,32209,insurance_outreach,9.79,9.8%,34657,False
3,32206,insurance_outreach,6.40,7.6%,17105,False
4,32208,insurance_outreach,5.84,8.2%,32699,False
5,32209,parks,5.20,5.2%,34657,False
6,32209,primary_care,4.83,4.8%,34657,False
7,32254,parks,4.67,4.8%,13927,False
8,32202,insurance_outreach,4.47,7.8%,6023,False
9,32206,food_access,4.43,5.3%,17105,False


---
## Stage 6 — Per-Category Drill-Down (US4)

Any service category can be isolated and visualised independently.

In [14]:
for category in ['healthcare', 'food_access', 'parks', 'insurance']:
    cat_df = filter_by_service_category(desert_scores, category)
    plot_category_view(cat_df, category=category)
    out = cfg.REPORTS_FIGURES_DIR / f'category_{category}_view.png'
    print(f"  {category:15s} — {len(cat_df)} ZIPs — saved → {out}")

2026-03-28 22:51:17,368 INFO [src.visualization] Saved category view → /Users/amitrajpurkar/workspace/pyprojects/unf_datathons/prob1-resource-desert/reports/figures/category_healthcare_view.png


  healthcare      — 35 ZIPs — saved → /Users/amitrajpurkar/workspace/pyprojects/unf_datathons/prob1-resource-desert/reports/figures/category_healthcare_view.png


2026-03-28 22:51:17,463 INFO [src.visualization] Saved category view → /Users/amitrajpurkar/workspace/pyprojects/unf_datathons/prob1-resource-desert/reports/figures/category_food_access_view.png


  food_access     — 35 ZIPs — saved → /Users/amitrajpurkar/workspace/pyprojects/unf_datathons/prob1-resource-desert/reports/figures/category_food_access_view.png


2026-03-28 22:51:17,574 INFO [src.visualization] Saved category view → /Users/amitrajpurkar/workspace/pyprojects/unf_datathons/prob1-resource-desert/reports/figures/category_parks_view.png


  parks           — 35 ZIPs — saved → /Users/amitrajpurkar/workspace/pyprojects/unf_datathons/prob1-resource-desert/reports/figures/category_parks_view.png


2026-03-28 22:51:17,670 INFO [src.visualization] Saved category view → /Users/amitrajpurkar/workspace/pyprojects/unf_datathons/prob1-resource-desert/reports/figures/category_insurance_view.png


  insurance       — 35 ZIPs — saved → /Users/amitrajpurkar/workspace/pyprojects/unf_datathons/prob1-resource-desert/reports/figures/category_insurance_view.png


---
## Summary

| Output | Location |
|---|---|
| Desert scores CSV | `reports/outputs/desert_scores.csv` |
| Intervention recommendations JSON | `reports/outputs/intervention_recommendations.json` |
| Choropleth map HTML | `reports/outputs/resource_desert_map.html` |
| Bar chart | `reports/figures/desert_scores_bar_chart.png` |
| Health scatter | `reports/figures/preventative_asset_vs_health_outcome.png` |
| Intervention heatmap | `reports/figures/intervention_impact_heatmap.png` |
| Category views (×4) | `reports/figures/category_{name}_view.png` |